# CYK Parser Success Rate Evaluation

This notebook evaluates the CYK parser success rate across all California committee hearings in the corpus.

In [1]:
from pathlib import Path
import subprocess
import zipfile

# This is where the unzipped corpus file is stored
CORPUS_FILE_PATH = 'DH2024_Corpus_Release/'
corpus_dir = Path(CORPUS_FILE_PATH)
zip_path = Path("digitaldemocracy-2015-2018/DH2024_Corpus_Release.zip")
repo_dir = Path("digitaldemocracy-2015-2018")

# Clone repository if not present
if not repo_dir.is_dir():
    print("Corpus directory not found. Cloning repository...")
    subprocess.run(
        ["git", "clone", "https://huggingface.co/datasets/iatpp/digitaldemocracy-2015-2018"],
        check=True,
    )
    print("Repository cloned successfully.")

# Extract zip file if corpus directory doesn't exist
if not corpus_dir.is_dir():
    if zip_path.exists():
        print(f"Extracting {zip_path}...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall('.')
        print("Extraction complete.")
    else:
        print(f"Error: {zip_path} not found!")
else:
    print(f"Corpus already extracted at {corpus_dir}")

Corpus already extracted at DH2024_Corpus_Release


## Load California Hearings

In [2]:
from src import HearingLoader, HearingTagger
from src.grammar.Tokenizer import Tokenizer
import time

# Initialize the HearingLoader with the corpus path
loader = HearingLoader(corpus_path='DH2024_Corpus_Release/')
tagger = HearingTagger()
tokenizer = Tokenizer()

print("Loading all committee hearings...")
start_time = time.time()
all_hearings = loader.load_all_committee_hearings()
load_time = time.time() - start_time
print(f"Total hearings loaded: {len(all_hearings)} (took {load_time:.1f}s)")

# Filter for California hearings only
ca_hearings = [h for h in all_hearings if h.state == 'CA']
print(f"California hearings: {len(ca_hearings)}")

# Filter out hearings with no bill discussed
ca_hearings_with_bills = [h for h in ca_hearings if h.bid != 'CA_NO BILL DISCUSSED']
print(f"California hearings with bills: {len(ca_hearings_with_bills)}")

Loading all committee hearings...
invalid literal for int() with base 10: 'hid'
invalid literal for int() with base 10: 'hid'
invalid literal for int() with base 10: 'hid'
Total hearings loaded: 8218 (took 10.9s)
California hearings: 8218
California hearings with bills: 7069


## Run Tagger and Evaluate CYK Parser (Combined)

To improve efficiency, we'll tag and parse in a single loop with progress tracking.

In [3]:
import numpy as np

print("Processing hearings (tagging + parsing)...")
print("This will take several minutes...\n")

parse_results = []
successful_parses = []
failed_parses = []
tagging_errors = []

total = len(ca_hearings_with_bills)
start_time = time.time()

for i, hearing in enumerate(ca_hearings_with_bills):
    # Progress update every 50 hearings
    if (i + 1) % 100 == 0:
        elapsed = time.time() - start_time
        rate = (i + 1) / elapsed
        remaining = (total - i - 1) / rate if rate > 0 else 0
        print(f"Processed {i + 1}/{total} hearings ({(i+1)/total*100:.1f}%)")
    
    try:
        # Tag the hearing
        tagged_hearing = tagger(hearing)
        
        # Try to parse
        try:
            parse_trees = list(tokenizer.get_all_parses_as_nltk_trees(tagged_hearing, max_parses=2) or [])
            parse_success = len(parse_trees) > 0
            
            parse_results.append(int(parse_success))
            
            if parse_success:
                successful_parses.append((hearing.hid, hearing.bid, len(parse_trees)))
            else:
                failed_parses.append((hearing.hid, hearing.bid))
                
        except Exception as e:
            # Parsing failed with exception
            parse_results.append(0)
            failed_parses.append((hearing.hid, hearing.bid))
            
    except Exception as e:
        # Tagging failed
        tagging_errors.append((hearing.hid, hearing.bid, str(e)))
        continue

total_time = time.time() - start_time
print(f"\nProcessing complete! Total time: {total_time:.1f}s ({total_time/60:.1f} minutes)")
print(f"Successfully processed: {len(parse_results)}/{total} hearings")
print(f"Tagging errors: {len(tagging_errors)}")

Processing hearings (tagging + parsing)...
This will take several minutes...

Processed 100/7069 hearings (1.4%)
Processed 200/7069 hearings (2.8%)
Processed 300/7069 hearings (4.2%)
Processed 400/7069 hearings (5.7%)
Processed 500/7069 hearings (7.1%)
Processed 600/7069 hearings (8.5%)
Processed 700/7069 hearings (9.9%)
Processed 800/7069 hearings (11.3%)
Processed 900/7069 hearings (12.7%)
Processed 1000/7069 hearings (14.1%)
Processed 1100/7069 hearings (15.6%)
Processed 1200/7069 hearings (17.0%)
Processed 1300/7069 hearings (18.4%)
Processed 1400/7069 hearings (19.8%)
Processed 1500/7069 hearings (21.2%)
Processed 1600/7069 hearings (22.6%)
Processed 1700/7069 hearings (24.0%)
Processed 1800/7069 hearings (25.5%)
Processed 1900/7069 hearings (26.9%)
Processed 2000/7069 hearings (28.3%)
Processed 2100/7069 hearings (29.7%)
Processed 2200/7069 hearings (31.1%)
Processed 2300/7069 hearings (32.5%)
Processed 2400/7069 hearings (34.0%)
Processed 2500/7069 hearings (35.4%)
Processed 260

In [6]:
tagging_errors

[(52363, 'CA_201720180AB1246', 'min() iterable argument is empty'),
 (254780, 'CA_201720180AB2970', '100929'),
 (254896,
  'CA_201720180AB1867',
  'not enough values to unpack (expected 3, got 2)'),
 (255466, 'CA_201720180SB910', 'min() iterable argument is empty'),
 (51884, 'CA_201720180SB28', 'min() iterable argument is empty'),
 (251277, 'CA_201720180AB1708', 'min() iterable argument is empty'),
 (53971, 'CA_201720180SB233', 'min() iterable argument is empty'),
 (172547, 'CA_201720180AB669', 'min() iterable argument is empty'),
 (254683, 'CA_201720180AB2152', 'min() iterable argument is empty'),
 (52301, 'CA_201720180SB490', 'min() iterable argument is empty'),
 (52301, 'CA_201720180SB789', 'min() iterable argument is empty'),
 (52301, 'CA_201720180SB482', 'min() iterable argument is empty'),
 (255685, 'CA_201720180AB2605', 'min() iterable argument is empty'),
 (52479, 'CA_201720180AB448', 'min() iterable argument is empty'),
 (52479, 'CA_201720180AB552', 'min() iterable argument is

## Results

In [4]:
total_hearings = len(parse_results)
successful_count = np.sum(parse_results)
failed_count = total_hearings - successful_count
success_rate = np.mean(parse_results) * 100 if parse_results else 0

print("=" * 80)
print("CYK Parser Statistics for 2015-2018 California Hearings")
print("=" * 80)
print(f"\nTotal hearings evaluated: {total_hearings}")
print(f"Successful parses:\t{successful_count}")
print(f"Failed parses:\t{failed_count}")
print(f"\nSuccess rate:\t{success_rate:.2f}%")
print("\n" + "=" * 80)

CYK Parser Statistics for 2015-2018 California Hearings

Total hearings evaluated: 7029
Successful parses:	3881
Failed parses:	3148

Success rate:	55.21%



## Detailed Statistics

In [7]:
successful_parses[:5]

[(52054, 'CA_201720180AB816', 1),
 (52054, 'CA_201720180AB547', 1),
 (52054, 'CA_201720180AB822', 1),
 (52054, 'CA_201720180AB677', 1),
 (52363, 'CA_201720180AB262', 1)]

In [5]:
print("\nSuccessful Parses Statistics:")
print(f"  Total: {len(successful_parses)}")

if successful_parses:
    parse_tree_counts = [count for _, _, count in successful_parses]
    print(f"  Average parse trees per hearing: {np.mean(parse_tree_counts):.2f}")
    print(f"  Min parse trees: {np.min(parse_tree_counts)}")
    print(f"  Max parse trees: {np.max(parse_tree_counts)}")
    
    # Show distribution of parse tree counts
    unique, counts = np.unique(parse_tree_counts, return_counts=True)
    print("\n  Parse tree count distribution:")
    for tree_count, freq in zip(unique, counts):
        print(f"    {tree_count} tree(s): {freq} hearings ({freq/len(successful_parses)*100:.1f}%)")

print(f"\nFailed Parses:")
print(f"  Total: {len(failed_parses)}")

# Show first 10 failed parses as examples
if failed_parses:
    print("\n  Examples of failed parses (first 10):")
    for i, (hid, bid) in enumerate(failed_parses[:10]):
        print(f"    {i+1}. Hearing ID: {hid}, Bill: {bid}")

if tagging_errors:
    print(f"\nTagging Errors:")
    print(f"  Total: {len(tagging_errors)}")
    print("\n  Examples (first 5):")
    for i, (hid, bid, error) in enumerate(tagging_errors[:5]):
        print(f"    {i+1}. Hearing ID: {hid}, Bill: {bid}")
        print(f"       Error: {error[:100]}..." if len(error) > 100 else f"       Error: {error}")


Successful Parses Statistics:
  Total: 3881
  Average parse trees per hearing: 1.00
  Min parse trees: 1
  Max parse trees: 1

  Parse tree count distribution:
    1 tree(s): 3881 hearings (100.0%)

Failed Parses:
  Total: 3148

  Examples of failed parses (first 10):
    1. Hearing ID: 52054, Bill: CA_201720180AB92
    2. Hearing ID: 52054, Bill: CA_201720180AB12
    3. Hearing ID: 52363, Bill: CA_201720180AB1223
    4. Hearing ID: 52773, Bill: CA_201720180AB1635
    5. Hearing ID: 52773, Bill: CA_201720180AB912
    6. Hearing ID: 52773, Bill: CA_201720180AB632
    7. Hearing ID: 53992, Bill: CA_201720180SB173
    8. Hearing ID: 54113, Bill: CA_201720180SB574
    9. Hearing ID: 54013, Bill: CA_201720180SJR6
    10. Hearing ID: 254631, Bill: CA_201720180AB2192

Tagging Errors:
  Total: 40

  Examples (first 5):
    1. Hearing ID: 52363, Bill: CA_201720180AB1246
       Error: min() iterable argument is empty
    2. Hearing ID: 254780, Bill: CA_201720180AB2970
       Error: 100929
    3